# 태양풍 속도 예측 — P11

**P9 노트북의 셀 0~16 을 한 글자도 고치지 않고 그대로 쓰고**, 그 뒤에 P11 레이어를 붙여
필요한 함수만 재정의한다. 검증된 경로(데이터 로드 · CH 추출 · 탄도 정렬 · 모델 · 지표)는
건드리지 않았다.

## 무엇이 달라졌나 (기준: P3 = 58.80)

| 레버 | 내용 | 왜 |
|---|---|---|
| **L1** | 무적합 균등가중 앙상블 (설정 × 시드 × epoch) | epoch 1 에서 꺾이는 고분산 추정기라 평균의 이득이 크다. **적합 파라미터 0개** — P6 의 NNLS(val 로 84개 적합)와 다른 물건이다 |
| **L2** | μ 보정 진짜면적 + 등각 경도 셀 | 픽셀 면적은 림에서 최대 1/2.3 로 눌린다. 자전만으로 CH 시계열이 가짜로 흔들린다. **crop 은 하지 않는다** — P10 이 −2.98 을 잃은 요인이다 |
| **L3** | CH 경계거리 θ_b · 형태 | WSA 의 θ_b 항. 같은 면적이라도 큰 홀 하나와 작은 홀 여럿은 전혀 다른 바람을 낸다 |
| **L4** | 탄도창 확장 (−8 ~ +5) | 도착할 스트림의 τ 는 알 수 없으므로 창을 넓혀 head 가 고르게 둔다 |
| **L5** | horizon별 수축 보정 (12 파라미터, train OOF 적합) | RMSE 최적 예측은 조건부 평균이다 |

## 실행 순서

1. (선택) `python p11_extract.py` 를 백그라운드로 돌려 `work/cache/p11a_*.npz` 를 만든다.
   없으면 **L3 는 자동으로 꺼지고** L2 근사판만 쓴다 — 노트북은 그대로 끝까지 돈다.
2. 위에서부터 순서대로 실행한다. **바꿀 것은 P11 설정 셀의 `STAGE` 한 줄뿐이다.**
3. 마지막 두 셀(재현성 검증 · 제출 점검)이 통과해야 제출한다.

> **CV 사다리는 돌리지 않는다.** 계측기 감도가 실제 2.56 차이를 0.013 으로 읽는 수준이라
> 하루치 GPU 를 정당화하지 못한다. 판정은 리더보드 제출로 한다.
> 셀 16 은 [L5] 가 쓰는 `FOLDS` 정의 때문에 남겨 둔 것이고 `run_cv` 는 호출하지 않는다.

> **재현성.** `model.pth` 하나에 멤버 K개의 state_dict · 정규화 통계 · 설정이 전부 들어간다.
> "재현성 검증" 셀이 그 파일만으로 `submission.csv` 를 다시 만들어 RMS 차이를 잰다.

## 1. 설정

In [1]:
from pathlib import Path
import gc, hashlib, json, math, os, random, shutil, time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F

SEED = 777
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT_CANDIDATES = [
    Path(os.getenv("SW_DATA_ROOT", "")) if os.getenv("SW_DATA_ROOT") else None,
    Path("public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public_dataset/competition_dataset_6h"),
    Path("public/public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public/public_dataset/competition_dataset_6h"),
    Path("dataset"), Path("/home/jovyan/dataset"),
]
DATA_ROOT = None
for candidate in DATA_ROOT_CANDIDATES:
    if candidate is not None and (candidate / "train/inputs.csv").exists():
        DATA_ROOT = candidate
        break
if DATA_ROOT is None:
    raise FileNotFoundError("데이터 경로 없음")

WORK_DIR = Path("work")
CACHE_ROOT = WORK_DIR / "cache"
OUTPUT_DIR = WORK_DIR / "outputs_p9"
SUBMISSION_DIR = Path("submission")
for directory in (CACHE_ROOT, OUTPUT_DIR, SUBMISSION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

IMAGE_COLUMNS = [f"image_{i:02d}" for i in range(20)]
WIND_COLUMNS = [f"wind_{i:02d}" for i in range(20)]
TARGET_COLUMNS = [f"target_{i:02d}" for i in range(12)]
HORIZONS = np.arange(1, 13) * 6
CHANNELS = ("193", "211")
AU_KM = 1.496e8
LAST_INDEX = 19.0                  # 윈도우 마지막 관측 시점(T0)의 인덱스

# ---- 코로나홀 추출 (P7 과 동일 — 캐시 그대로 재사용) ----------------------
CH_CODE_VERSION = "p7a"
FINE_GRID = (12, 30)
CH_CUTS = (0.30, 0.45, 0.60)
BRIGHT_CUT = 1.60
DISK_MARGIN = 0.95
PER_FRAME_DISK = True

# ---- 사용할 격자 / 레벨 ---------------------------------------------------
CH_GRID = (6, 3)
FOLD_LATITUDE = True
USE_LEVELS = ("dark0.45", "bright")

# ---- 탄도 정렬 ------------------------------------------------------------
REFERENCE_TRANSIT_HOURS = 108.0    # EDA 경험적 최적 지연
TRANSIT_SPEEDS = (315.0, 345.0, 385.0, 435.0, 500.0, 600.0)   # 고정 모드에서만 사용
BALLISTIC_OFFSETS = (-4, -2, 0, 2)         # 탄도창 샘플 시점 (6h 스텝)
GATHER_OFFSETS = (-6, -4, -2, 0, 2, 6)     # 적응형 gather 샘플 시점
BALLISTIC_SOURCE = "window"        # "window" = 기준 τ ± offset / "speeds" = 속도별 점 샘플
BALLISTIC_LAT = "profile"          # "profile" = 위도 전부 / "equator" = 적도 1행
BALLISTIC_LON = "all"              # [R1] "all" = 경도 전부 / "central" = 중앙자오선 1열
USE_BALLISTIC_WINDOW = True

# ---- [R2] 적응형 전달 시간 ------------------------------------------------
ADAPTIVE_AREA = True               # 탄도창 인덱스를 샘플별 관측 속도로
ADAPTIVE_GATHER = True             # gather 인덱스를 샘플별 관측 속도로
USE_TRANSIT_FLAGS = True           # 관측창 이탈 플래그를 head 에 투입
SPEED_FLOOR, SPEED_CEIL = 280.0, 800.0
FLAG_DIM = 4                       # (미래이탈, 과거이탈, 이탈량+, 이탈량-)

# ---- 모델 -----------------------------------------------------------------
USE_CH_GATHER = True
CH_HIDDEN = 64
CH_BIDIRECTIONAL = True
GATHER_DIM = 16
HORIZON_EMBED = 8
DROPOUT = 0.4
VERBOSE_MODEL = True

# ---- 학습 -----------------------------------------------------------------
BATCH_SIZE = 64
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
GRAD_CLIP = 1.0
LOSS_EPSILON = 1e-8
LOSS_SCALE = 100.0
CV_EPOCHS = 30
AUGMENT = True
AUG_CH_NOISE = 0.05

# ---- 배치 공급 -------------------------------------------------------------
# 데이터가 전부 메모리 위 numpy 배열이라 I/O 가 없다. torch DataLoader 는
# train_run 호출마다 워커 프로세스를 새로 띄워서, 이 규모에서는 학습보다 오버헤드가 크다.
# 전 배열을 DEVICE 에 한 번 올려두고 인덱스 슬라이스로 배치를 만든다.
BATCHER_LIMIT_MB = 3000    # 이보다 크면 CPU 에 두고 배치마다 옮긴다

# ---- [C][D] 계측기 --------------------------------------------------------
RUN_FULL_LADDER = False   # 기본은 빠른 A/B 만. 귀속(어느 변경이 효과였나)이 필요할 때 True
QUICK_SEEDS = (777, 778)  # 대응 비교라 2개로 충분하다 (아래 설명)
N_FOLDS = 5
FOLD_MODE = "block"       # [D] "block"=시간 연속 블록 / "balanced"=P7 / "forward"=전진 검증
CV_SEEDS = (777, 778, 779)     # [C] 전체 사다리용 시드
LADDER_FOLDS = 3          # 사다리는 앞 3폴드만 (시간 절약). 확정 검증은 전 폴드
EPOCH_SMOOTH = 3          # [B] epoch 곡선 이동평균 창
NOISE_SIGMA = 2.0         # [C] 판정 임계 = NOISE_SIGMA * 결합 표준오차

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

print("PyTorch:", torch.__version__, "| device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("data:", DATA_ROOT.resolve())

PyTorch: 2.5.1+cu124 | device: cuda
GPU: NVIDIA A100-SXM4-40GB
data: /home/jovyan/public_dataset/competition_dataset_6h


## 2. 데이터 로드 · 시간축 복원

P7 과 동일하다. `inputs.csv` 에 타임스탬프가 없고 행 순서도 시간 순서가 아니므로,
한 행의 `image_00..image_19` 가 연속 20시점이라는 사실만으로 프레임 사슬을 복원한다.

**P9 에서 이 사슬의 역할이 하나 늘었다.** P7 에서는 "폴드 경계"였지만,
P9 의 `FOLD_MODE="block"` 에서는 **사슬 순서가 곧 시간 순서**라는 가정 위에서 폴드를 자른다.
아래 셀 마지막에서 그 가정을 파일명 정렬로 점검한다.

In [2]:
train_inputs = pd.read_csv(DATA_ROOT / "train/inputs.csv")
train_targets_frame = pd.read_csv(DATA_ROOT / "train/targets.csv")
val_inputs = pd.read_csv(DATA_ROOT / "validation/inputs.csv")
val_targets_frame = pd.read_csv(DATA_ROOT / "validation/targets.csv")
test_inputs = pd.read_csv(DATA_ROOT / "test/inputs.csv")
assert train_inputs.sample_id.tolist() == train_targets_frame.sample_id.tolist()
assert val_inputs.sample_id.tolist() == val_targets_frame.sample_id.tolist()


def fill_wind(inputs):
    wind = inputs[WIND_COLUMNS].to_numpy(np.float64)
    valid = np.isfinite(wind).astype(np.float32)
    frame = pd.DataFrame(wind).ffill(axis=1).bfill(axis=1)
    filled = frame.to_numpy(np.float32)
    return np.nan_to_num(filled, nan=float(np.nanmedian(wind))), valid


train_wind, train_wind_valid = fill_wind(train_inputs)
val_wind, val_wind_valid = fill_wind(val_inputs)
test_wind, test_wind_valid = fill_wind(test_inputs)
train_targets = train_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)
val_targets = val_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)


def reconstruct_frame_chains(inputs):
    """이미지 파일명만으로 프레임 시간축을 복원한다. 행 순서에 의존하지 않는다."""
    images = inputs[IMAGE_COLUMNS].to_numpy()
    successor, predecessor, conflicts = {}, {}, 0
    for row in images:
        for current, following in zip(row[:-1], row[1:]):
            if successor.setdefault(current, following) != following:
                conflicts += 1
            if predecessor.setdefault(following, current) != current:
                conflicts += 1
    names = set(images.ravel().tolist())
    chains, visited = [], set()
    for head in sorted(names - set(predecessor)):
        chain, node = [], head
        while node is not None and node not in visited:
            visited.add(node); chain.append(node); node = successor.get(node)
        chains.append(chain)
    assert conflicts == 0 and not (names - visited), "사슬 복원 실패"
    return chains


def sample_chain_index(inputs, chains):
    position = {name: (c, o)
                for c, chain in enumerate(chains) for o, name in enumerate(chain)}
    first = inputs[IMAGE_COLUMNS[0]].to_numpy()
    chain_id = np.array([position[n][0] for n in first])
    offset = np.array([position[n][1] for n in first])
    return chain_id, offset


TRAIN_CHAINS = reconstruct_frame_chains(train_inputs)
VAL_CHAINS = reconstruct_frame_chains(val_inputs)
TEST_CHAINS = reconstruct_frame_chains(test_inputs)
TRAIN_CHAIN_ID, TRAIN_OFFSET = sample_chain_index(train_inputs, TRAIN_CHAINS)

train_files = [n for chain in TRAIN_CHAINS for n in chain]
val_files = [n for chain in VAL_CHAINS for n in chain]
test_files = [n for chain in TEST_CHAINS for n in chain]
train_map = {n: i for i, n in enumerate(train_files)}
val_map = {n: i for i, n in enumerate(val_files)}
test_map = {n: i for i, n in enumerate(test_files)}

print(f"train {len(train_inputs):,} 샘플 / 사슬 {len(TRAIN_CHAINS)}개 / 고유 이미지 {len(train_files):,}")
print(f"val   {len(val_inputs):,} 샘플 / 사슬 {len(VAL_CHAINS)}개 / 고유 이미지 {len(val_files):,}")
print(f"test  {len(test_inputs):,} 샘플 / 사슬 {len(TEST_CHAINS)}개 / 고유 이미지 {len(test_files):,}")
print(f"\ntrain 사슬별 샘플 수: {np.bincount(TRAIN_CHAIN_ID).tolist()}")

# --- [D] 사슬 순서 == 시간 순서 가정 점검 ---------------------------------
heads = [chain[0] for chain in TRAIN_CHAINS]
CHAIN_ORDER_IS_TIME = heads == sorted(heads)
print(f"\n[D] 사슬 머리 파일명이 정렬 순서인가: {CHAIN_ORDER_IS_TIME}")
if not CHAIN_ORDER_IS_TIME:
    print("    -> 파일명이 시간 인코딩이 아닐 수 있다. FOLD_MODE='block' 의 시간 가정을 확인할 것.")
print("    사슬 머리 5개:", heads[:5])
print("    사슬 꼬리 5개:", [chain[-1] for chain in TRAIN_CHAINS][-5:])

train 9,607 샘플 / 사슬 28개 / 고유 이미지 10,139
val   1,199 샘플 / 사슬 11개 / 고유 이미지 1,408
test  3,868 샘플 / 사슬 13개 / 고유 이미지 4,115

train 사슬별 샘플 수: [629, 96, 961, 564, 285, 251, 169, 792, 578, 528, 149, 316, 100, 363, 14, 316, 232, 76, 747, 173, 161, 108, 138, 54, 161, 33, 961, 652]

[D] 사슬 머리 파일명이 정렬 순서인가: True
    사슬 머리 5개: ['image_000913.png', 'image_001424.png', 'image_001512.png', 'image_001602.png', 'image_001739.png']
    사슬 꼬리 5개: ['image_002569.png', 'image_005859.png', 'image_007913.png', 'image_009995.png', 'image_009770.png']


## 3. 코로나홀 추출

**P7 과 완전히 동일하다.** `CH_CODE_VERSION` 을 그대로 두었으므로 P7 이 만든 캐시가 재사용된다.
(추출 로직을 손대면 반드시 이 값을 올릴 것.)

In [3]:
FINE_LAT, FINE_LON = FINE_GRID
FINE_CELLS = FINE_LAT * FINE_LON
N_LEVELS = len(CH_CUTS) + 1
LEVEL_NAMES = [f"dark{c}" for c in CH_CUTS] + ["bright"]


def load_pair(split, name):
    planes = []
    for channel in CHANNELS:
        with Image.open(DATA_ROOT / split / channel / name) as image:
            planes.append(np.asarray(image.convert("L"), dtype=np.float32))
    return np.stack(planes)


def detect_disk(frame):
    """플레어에 둔감한 원반 검출. 배경과 원반 내부의 중간값을 임계로 쓴다."""
    plane = frame.mean(axis=0)
    background = np.percentile(plane, 2.0)
    interior = np.percentile(plane, 70.0)
    mask = plane > background + 0.35 * (interior - background)
    ys, xs = np.nonzero(mask)
    return float(ys.mean()), float(xs.mean()), float(math.sqrt(mask.sum() / math.pi))


_geometry_cache = {}
GEOMETRY_CACHE_LIMIT = 96


def cell_geometry(side, center_y, center_x, radius):
    key = (side, round(center_y), round(center_x), round(radius))
    entry = _geometry_cache.get(key)
    if entry is None:
        _, cy, cx, r = key
        grid_y, grid_x = np.mgrid[0:side, 0:side].astype(np.float32)
        disk = np.sqrt((grid_y - cy) ** 2 + (grid_x - cx) ** 2) <= r
        lat = np.clip((grid_y - (cy - r)) / (2 * r) * FINE_LAT, 0, FINE_LAT - 1e-4)
        lon = np.clip((grid_x - (cx - r)) / (2 * r) * FINE_LON, 0, FINE_LON - 1e-4)
        cell = (lat.astype(np.int32) * FINE_LON + lon.astype(np.int32))[disk]
        entry = (disk, cell)
        _geometry_cache[key] = entry
        while len(_geometry_cache) > GEOMETRY_CACHE_LIMIT:
            _geometry_cache.pop(next(iter(_geometry_cache)))
    return entry


def extract_split(split, filenames):
    area = np.zeros((len(filenames), N_LEVELS, FINE_CELLS), np.float32)
    geometry = np.zeros((len(filenames), 4), np.float32)
    fixed = None
    if not PER_FRAME_DISK:
        sample = np.unique(np.linspace(0, len(filenames) - 1, 200).astype(int))
        measured = np.array([detect_disk(load_pair(split, filenames[i])) for i in sample])
        fixed = tuple(np.median(measured, axis=0))
    started = time.perf_counter()
    for index, name in enumerate(filenames):
        frame = load_pair(split, name)
        center_y, center_x, radius = fixed if fixed is not None else detect_disk(frame)
        disk, cell = cell_geometry(frame.shape[-1], center_y, center_x, radius * DISK_MARGIN)
        on_disk = frame[:, disk]
        total = max(on_disk.shape[1], 1)
        geometry[index] = (center_y, center_x, radius, total)
        median = np.median(on_disk[:, ::4], axis=1, keepdims=True)
        normalized = on_disk / np.maximum(median, 1e-3)
        for level, cut in enumerate(CH_CUTS):
            selection = np.logical_and(normalized[0] <= cut, normalized[1] <= cut)
            area[index, level] = np.bincount(cell[selection], minlength=FINE_CELLS) / total
        selection = np.logical_and(normalized[0] >= BRIGHT_CUT, normalized[1] >= BRIGHT_CUT)
        area[index, -1] = np.bincount(cell[selection], minlength=FINE_CELLS) / total
        if (index + 1) % 2000 == 0 or index + 1 == len(filenames):
            print(f"  {split} {index + 1}/{len(filenames)} "
                  f"({time.perf_counter() - started:.0f}s)", flush=True)
    return area, geometry


def cached_extract(split, filenames):
    tag = (f"{CH_CODE_VERSION}_{FINE_LAT}x{FINE_LON}"
           f"_c{'-'.join(str(c) for c in CH_CUTS)}_b{BRIGHT_CUT}_m{DISK_MARGIN}"
           f"_{'perframe' if PER_FRAME_DISK else 'fixed'}")
    area_path = CACHE_ROOT / f"ch_{split}_{tag}.npy"
    geometry_path = CACHE_ROOT / f"geom_{split}_{tag}.npy"
    if area_path.exists() and geometry_path.exists():
        area = np.load(area_path)
        if area.shape == (len(filenames), N_LEVELS, FINE_CELLS):
            print(f"캐시 재사용: {area_path.name}")
            return area, np.load(geometry_path)
    area, geometry = extract_split(split, filenames)
    np.save(area_path, area); np.save(geometry_path, geometry)
    return area, geometry


train_area, train_geometry = cached_extract("train", train_files)
val_area, val_geometry = cached_extract("validation", val_files)
test_area, test_geometry = cached_extract("test", test_files)

for name, geometry in [("train", train_geometry), ("val", val_geometry), ("test", test_geometry)]:
    radius = geometry[:, 2]
    print(f"{name:5s} 반지름 {radius.mean():.1f} ± {radius.std():.1f}px "
          f"({radius.std()/radius.mean():.2%}), 원반 픽셀 {geometry[:, 3].mean():,.0f}")

캐시 재사용: ch_train_p7a_12x30_c0.3-0.45-0.6_b1.6_m0.95_perframe.npy
캐시 재사용: ch_validation_p7a_12x30_c0.3-0.45-0.6_b1.6_m0.95_perframe.npy
캐시 재사용: ch_test_p7a_12x30_c0.3-0.45-0.6_b1.6_m0.95_perframe.npy
train 반지름 228.7 ± 10.7px (4.68%), 원반 픽셀 148,544
val   반지름 227.7 ± 10.6px (4.67%), 원반 픽셀 147,313
test  반지름 231.3 ± 11.2px (4.83%), 원반 픽셀 151,982


## 4. 격자 집계 · 탄도 정렬 — **[R1] 경도 확장 · [R2] 적응형 전달 시간**

### [R2] 적응형 전달 시간

P7 은 모든 샘플에 같은 지연 108h 를 적용했다. 108h 는 **모집단 평균**이고,
고속 스트림 샘플의 실제 전달 시간은 70h 아래로 떨어진다.

관측된 최근 4스텝 평균 속도 $\hat v$ 로 샘플별 지연을 만든다.

$$\hat\tau_i = \mathrm{TAU\_SCALE}\cdot\frac{1\,\mathrm{AU}}{\hat v_i},\qquad
\mathrm{TAU\_SCALE}=\frac{108\,\mathrm{h}}{1\mathrm{AU}/\bar v_{\text{train}}}$$

`TAU_SCALE` 은 **train 평균 속도에서 정확히 108h 가 되도록** 맞춘 보정 계수다.
EDA 의 경험적 최적을 유지한 채 샘플별로만 흔든다.

인덱스는 $\mathrm{idx}_{i,h} = 19 + (6h - \hat\tau_i)/6$ 이고, $[0,19]$ 밖으로 나가면 clip 한다.

### [R2] 관측창 이탈 플래그

**고정 τ 에서는 clip 여부가 horizon 만의 함수라 horizon embedding 과 중복이어서 무의미했다.**
적응형이 되면 같은 horizon 이라도 샘플마다 clip 여부가 달라지므로 비로소 정보가 된다.
`(미래이탈 여부, 과거이탈 여부, 이탈량+, 이탈량-)` 4개를 head 에 직접 넣는다.

### [R1] 탄도 경도 확장

`BALLISTIC_LON="all"` 이면 위도 프로파일 × **전 경도**를 뽑는다.
gather 경로(양방향 GRU 은닉)는 시퀀스 전체를 섞은 값이라 시간 국소성이 흐려지는데,
원시 셀 값을 소스 시각에서 직접 뽑으면 날카로운 신호가 하나 더 들어간다.

In [4]:
def aggregate(area, grid_lat, grid_lon, fold):
    """(n, L, FINE_CELLS) -> (n, L, cells). 면적이 원반 대비 비율이라 단순 합이 정확하다."""
    block = area.reshape(len(area), N_LEVELS, FINE_LAT, FINE_LON)
    block = block.reshape(len(area), N_LEVELS, grid_lat, FINE_LAT // grid_lat,
                          grid_lon, FINE_LON // grid_lon).sum(axis=(3, 5))
    if fold:
        block = block + block[:, :, ::-1, :]
        block = block[:, :, : (grid_lat + 1) // 2, :]
    return np.ascontiguousarray(block.reshape(len(area), N_LEVELS, -1), dtype=np.float32)


def image_index_matrix(inputs, image_map):
    return np.asarray([[image_map[n] for n in row]
                       for row in inputs[IMAGE_COLUMNS].itertuples(index=False, name=None)],
                      dtype=np.int64)


train_index_matrix = image_index_matrix(train_inputs, train_map)
val_index_matrix = image_index_matrix(val_inputs, val_map)
test_index_matrix = image_index_matrix(test_inputs, test_map)

# TAU_SCALE 은 train 전용 스칼라. EDA 의 108h 를 train 평균 속도에 고정시킨다.
TAU_REF_SPEED = float(train_wind.mean())
TAU_SCALE = REFERENCE_TRANSIT_HOURS / (AU_KM / TAU_REF_SPEED / 3600.0)
print(f"[R2] train 평균 속도 {TAU_REF_SPEED:.1f} km/s -> 순수 전달시간 "
      f"{AU_KM / TAU_REF_SPEED / 3600.0:.1f}h, TAU_SCALE={TAU_SCALE:.3f} "
      f"(보정 후 {REFERENCE_TRANSIT_HOURS:.0f}h)")


def estimate_tau(wind):
    """샘플별 전달 지연(시간). (n,)"""
    speed = np.clip(wind[:, -4:].mean(axis=1), SPEED_FLOOR, SPEED_CEIL)
    return TAU_SCALE * AU_KM / speed / 3600.0


def reference_index_raw(wind, adaptive):
    """클리핑 전 기준 인덱스. (n, 12)"""
    tau = estimate_tau(wind) if adaptive else np.full(len(wind), REFERENCE_TRANSIT_HOURS)
    return LAST_INDEX + (HORIZONS[None, :] - tau[:, None]) / 6.0


def offset_indices(wind, offsets, adaptive):
    """기준 인덱스에 오프셋을 더해 클리핑. (n, 12, K)"""
    raw = reference_index_raw(wind, adaptive)[:, :, None] \
        + np.asarray(offsets, np.float64)[None, None, :]
    return np.clip(raw, 0.0, LAST_INDEX).astype(np.float32)


def fixed_speed_indices(n_samples, speeds):
    """P3/P7 방식: 고정 속도별 점 샘플. (n, 12, S)"""
    table = np.zeros((12, len(speeds)), np.float64)
    for h in range(12):
        for s, speed in enumerate(speeds):
            table[h, s] = np.clip(
                LAST_INDEX + ((h + 1) * 6.0 - AU_KM / speed / 3600.0) / 6.0,
                0.0, LAST_INDEX)
    return np.repeat(table[None].astype(np.float32), n_samples, axis=0)


def area_indices(wind):
    if BALLISTIC_SOURCE == "window":
        return offset_indices(wind, BALLISTIC_OFFSETS, ADAPTIVE_AREA)
    return fixed_speed_indices(len(wind), TRANSIT_SPEEDS)


def gather_indices(wind):
    if ADAPTIVE_GATHER:
        return offset_indices(wind, GATHER_OFFSETS, True)
    return fixed_speed_indices(len(wind), TRANSIT_SPEEDS)


def transit_flags(wind):
    """[R2] 관측창 이탈 플래그. (n, 12, FLAG_DIM)"""
    raw = reference_index_raw(wind, ADAPTIVE_AREA)
    over = np.clip(raw - LAST_INDEX, 0.0, None) / 8.0
    under = np.clip(-raw, 0.0, None) / 8.0
    return np.stack([(raw > LAST_INDEX).astype(np.float32),
                     (raw < 0.0).astype(np.float32),
                     over.astype(np.float32),
                     under.astype(np.float32)], axis=2).astype(np.float32)


def pick_per_sample(sequence, index):
    """sequence (n, 20, D) 를 샘플별 실수 인덱스 index (n, K) 에서 선형보간. -> (n, K, D)"""
    lower = np.floor(index).astype(np.int64)
    upper = np.minimum(lower + 1, int(LAST_INDEX))
    weight = (index - lower).astype(np.float32)[:, :, None]
    rows = np.arange(len(sequence))[:, None]
    return sequence[rows, lower] * (1.0 - weight) + sequence[rows, upper] * weight


def flatten_ch(grid, indexes):
    return grid[indexes].reshape(len(indexes), 20, N_USED_LEVELS * N_CELLS).astype(np.float32)


def ballistic_columns():
    """[R1] 탄도 샘플에 쓸 셀 번호."""
    lons = list(range(GRID_LON)) if BALLISTIC_LON == "all" else [CENTRAL_LON]
    return [lat * GRID_LON + lon for lat in AREA_LAT_ROWS for lon in lons], len(lons)


def ballistic_area(grid, indexes, index):
    """탄도 소스 시각의 코로나홀 면적. index (n, 12, K) -> (n, 12, K*D)"""
    columns, _ = ballistic_columns()
    sequence = grid[indexes][:, :, :, columns]
    sequence = sequence.reshape(len(indexes), 20, -1)
    n_samples, n_horizon, n_offset = index.shape
    picked = pick_per_sample(sequence, index.reshape(n_samples, n_horizon * n_offset))
    return picked.reshape(n_samples, n_horizon, -1).astype(np.float32)


def build_ballistic(grid, indexes, wind):
    return ballistic_area(grid, indexes, area_indices(wind))


def configure(**overrides):
    """설정을 바꾸고 파생 상수를 다시 만든다. ablation 은 이 함수로만 한다."""
    globals().update(overrides)
    global GRID_LAT, GRID_LON, USED_LAT, N_CELLS, CENTRAL_LON, EQUATOR_ROW
    global LEVEL_INDEX, N_USED_LEVELS, CH_SEQ_DIM
    global train_grid, val_grid, test_grid
    global AREA_LAT_ROWS, N_AREA_OFFSETS, BALLISTIC_DIM, N_GATHER

    GRID_LAT, GRID_LON = CH_GRID
    assert FINE_LAT % GRID_LAT == 0 and FINE_LON % GRID_LON == 0
    USED_LAT = (GRID_LAT + 1) // 2 if FOLD_LATITUDE else GRID_LAT
    N_CELLS = USED_LAT * GRID_LON
    CENTRAL_LON = GRID_LON // 2
    EQUATOR_ROW = USED_LAT - 1 if FOLD_LATITUDE else GRID_LAT // 2
    LEVEL_INDEX = [LEVEL_NAMES.index(name) for name in USE_LEVELS]
    N_USED_LEVELS = len(LEVEL_INDEX)
    CH_SEQ_DIM = N_USED_LEVELS * N_CELLS

    train_grid = aggregate(train_area, GRID_LAT, GRID_LON, FOLD_LATITUDE)[:, LEVEL_INDEX]
    val_grid = aggregate(val_area, GRID_LAT, GRID_LON, FOLD_LATITUDE)[:, LEVEL_INDEX]
    test_grid = aggregate(test_area, GRID_LAT, GRID_LON, FOLD_LATITUDE)[:, LEVEL_INDEX]

    AREA_LAT_ROWS = list(range(USED_LAT)) if BALLISTIC_LAT == "profile" else [EQUATOR_ROW]
    N_AREA_OFFSETS = (len(BALLISTIC_OFFSETS) if BALLISTIC_SOURCE == "window"
                      else len(TRANSIT_SPEEDS))
    _, n_lons = ballistic_columns()
    BALLISTIC_DIM = N_AREA_OFFSETS * len(AREA_LAT_ROWS) * n_lons * N_USED_LEVELS
    N_GATHER = len(GATHER_OFFSETS) if ADAPTIVE_GATHER else len(TRANSIT_SPEEDS)


configure()
print(f"\n격자 {GRID_LAT}x{GRID_LON} -> 셀 {N_CELLS} (적도 대칭 접기 {FOLD_LATITUDE}), 레벨 {USE_LEVELS}")
print(f"ch_seq {CH_SEQ_DIM} / 탄도 {BALLISTIC_DIM} (경도 {BALLISTIC_LON}) / "
      f"gather {N_GATHER} / 플래그 {FLAG_DIM if USE_TRANSIT_FLAGS else 0}")

_tau = estimate_tau(train_wind)
print(f"\n[R2] 샘플별 전달 지연 분포: {np.percentile(_tau, [5, 25, 50, 75, 95]).round(1)} h "
      f"(5/25/50/75/95%)  — P7 은 전 샘플 {REFERENCE_TRANSIT_HOURS:.0f}h 고정")
_raw = reference_index_raw(train_wind, True)
print(f"[R2] 관측창 이탈 비율 (idx>19): "
      f"{[f'{h}h {(_raw[:, i] > LAST_INDEX).mean():.1%}' for i, h in enumerate(HORIZONS)][6:]}")
print("\n[R2] 기준 인덱스 분위수 (19 = 마지막 관측)")
print(pd.DataFrame(np.percentile(np.clip(_raw, 0, LAST_INDEX), [10, 50, 90], axis=0).T,
                   index=[f"{h}h" for h in HORIZONS], columns=["p10", "p50", "p90"]).round(2))

[R2] train 평균 속도 414.6 km/s -> 순수 전달시간 100.2h, TAU_SCALE=1.077 (보정 후 108h)

격자 6x3 -> 셀 9 (적도 대칭 접기 True), 레벨 ('dark0.45', 'bright')
ch_seq 18 / 탄도 72 (경도 all) / gather 6 / 플래그 4

[R2] 샘플별 전달 지연 분포: [ 75.6  97.  113.2 128.1 144.9] h (5/25/50/75/95%)  — P7 은 전 샘플 108h 고정
[R2] 관측창 이탈 비율 (idx>19): ['42h 0.0%', '48h 0.0%', '54h 0.0%', '60h 0.0%', '66h 0.7%', '72h 3.1%']

[R2] 기준 인덱스 분위수 (19 = 마지막 관측)
      p10    p50    p90
6h   0.00   1.13   6.21
12h  0.00   2.13   7.21
18h  0.00   3.13   8.21
24h  0.00   4.13   9.21
30h  0.75   5.13  10.21
36h  1.75   6.13  11.21
42h  2.75   7.13  12.21
48h  3.75   8.13  13.21
54h  4.75   9.13  14.21
60h  5.75  10.13  15.21
66h  6.75  11.13  16.21
72h  7.75  12.13  17.21


## 5. 정규화 통계 · Dataset

CV 폴드마다 **그 폴드의 학습 행에서만** 통계를 다시 낸다. 최종 학습에서는 train 전체를 쓴다.

Dataset 이 `gather_idx` 와 `flags` 를 함께 실어 보낸다 — 적응형 인덱스는 샘플마다 다르므로
모델 버퍼에 상수로 넣을 수 없다.

**CH 노이즈 증강을 곱셈에서 덧셈으로 바꿨다.** 표준화된 값에 곱셈 노이즈를 주면
평균 근처 셀은 노이즈가 0, 극단값 셀은 최대가 되어 의도와 반대로 작동한다.

In [5]:
STAT_NAMES = ["last", "mean4", "mean", "std", "min", "max", "slope", "last_minus_mean4", "range"]
NUM_STATS = len(STAT_NAMES)
_TIME_CENTERED = np.arange(20, dtype=np.float32) - 9.5
_TIME_DENOM = float((_TIME_CENTERED ** 2).sum())


def build_wind_stats(wind):
    last = wind[:, -1]
    mean4 = wind[:, -4:].mean(axis=1)
    slope = (wind - wind.mean(axis=1, keepdims=True)) @ _TIME_CENTERED / _TIME_DENOM
    return np.stack([last, mean4, wind.mean(axis=1), wind.std(axis=1), wind.min(axis=1),
                     wind.max(axis=1), slope, last - mean4,
                     wind.max(axis=1) - wind.min(axis=1)], axis=1).astype(np.float32)


def fit_stats(rows):
    """주어진 train 행에서만 정규화/복원 통계를 산출한다."""
    wind = train_wind[rows]
    targets = train_targets[rows]
    ch_seq = flatten_ch(train_grid, train_index_matrix[rows]).reshape(-1, CH_SEQ_DIM)
    ballistic = build_ballistic(train_grid, train_index_matrix[rows], wind
                                ).reshape(-1, BALLISTIC_DIM)
    statistics = build_wind_stats(wind)
    residual = targets - wind[:, -1:]
    return {
        "wind_mean": float(wind.mean()), "wind_std": float(wind.std() + 1e-6),
        "diff_std": float(np.diff(wind, axis=1, prepend=wind[:, :1]).std() + 1e-6),
        "stats_mean": statistics.mean(axis=0), "stats_std": statistics.std(axis=0) + 1e-6,
        "ch_mean": ch_seq.mean(axis=0), "ch_std": ch_seq.std(axis=0) + 1e-8,
        "ballistic_mean": ballistic.mean(axis=0), "ballistic_std": ballistic.std(axis=0) + 1e-8,
        "residual_mean": residual.mean(axis=0), "residual_std": residual.std(axis=0) + 1e-6,
        "clip_low": float(targets.min() * 0.95), "clip_high": float(targets.max() * 1.05),
        "target_mean": targets.mean(axis=0),
    }


BATCH_KEYS = ("wind_seq", "wind_stats", "ch_seq", "ballistic",
              "gather_idx", "flags", "last_wind")


def build_arrays(inputs, index_matrix, wind, wind_valid, grid, stats, targets=None):
    """모델 입력 배열 일체를 만든다 (numpy)."""
    arrays = {
        "wind_seq": np.stack([
            (wind - stats["wind_mean"]) / stats["wind_std"],
            np.diff(wind, axis=1, prepend=wind[:, :1]) / stats["diff_std"],
            wind_valid], axis=2).astype(np.float32),
        "wind_stats": ((build_wind_stats(wind) - stats["stats_mean"])
                       / stats["stats_std"]).astype(np.float32),
        "ch_seq": ((flatten_ch(grid, index_matrix) - stats["ch_mean"])
                   / stats["ch_std"]).astype(np.float32),
        "ballistic": ((build_ballistic(grid, index_matrix, wind) - stats["ballistic_mean"])
                      / stats["ballistic_std"]).astype(np.float32),
        "gather_idx": gather_indices(wind),
        "flags": transit_flags(wind),
        "last_wind": np.ascontiguousarray(wind[:, -1]).astype(np.float32),
    }
    if targets is not None:
        arrays["target"] = targets.astype(np.float32)
    arrays["sample_ids"] = inputs.sample_id.to_numpy()
    return arrays


class Batcher:
    """배열을 한 번만 텐서로 올려두고 인덱스 슬라이스로 배치를 낸다.

    증강(CH 노이즈)은 배치마다 GPU 에서 더한다 — 표준화된 값이므로 덧셈이 맞다
    (P7 은 곱셈이라 평균 근처 셀은 노이즈 0, 극단값 셀만 흔들려 방향이 반대였다).
    """

    def __init__(self, arrays, shuffle=False, seed=SEED, training=False):
        self.keys = [k for k in BATCH_KEYS + ("target",) if k in arrays]
        total_mb = sum(arrays[k].nbytes for k in self.keys) / 1024 ** 2
        self.device = DEVICE if total_mb <= BATCHER_LIMIT_MB else torch.device("cpu")
        self.tensors = {k: torch.as_tensor(arrays[k]).to(self.device) for k in self.keys}
        self.sample_ids = arrays["sample_ids"]
        self.count = len(self.sample_ids)
        self.shuffle = shuffle
        self.training = training
        self.generator = torch.Generator().manual_seed(seed)
        self.megabytes = total_mb

    def __len__(self):
        return (self.count + BATCH_SIZE - 1) // BATCH_SIZE

    def __iter__(self):
        order = (torch.randperm(self.count, generator=self.generator) if self.shuffle
                 else torch.arange(self.count))
        order = order.to(self.device)
        for start in range(0, self.count, BATCH_SIZE):
            selection = order[start:start + BATCH_SIZE]
            batch = {k: v.index_select(0, selection) for k, v in self.tensors.items()}
            if self.training and AUGMENT and AUG_CH_NOISE > 0:
                for key in ("ch_seq", "ballistic"):
                    batch[key] = batch[key] + torch.randn(
                        batch[key].shape, device=self.device) * AUG_CH_NOISE
            yield batch

    def release(self):
        self.tensors.clear()


def make_batcher(inputs, index_matrix, wind, wind_valid, grid, stats, targets=None,
                 shuffle=False, seed=SEED, training=False):
    arrays = build_arrays(inputs, index_matrix, wind, wind_valid, grid, stats, targets)
    return Batcher(arrays, shuffle=shuffle, seed=seed, training=training)


_probe_stats = fit_stats(np.arange(len(train_inputs)))
_probe_batcher = make_batcher(train_inputs, train_index_matrix, train_wind, train_wind_valid,
                              train_grid, _probe_stats, train_targets, shuffle=True)
print(f"입력 배열 {_probe_batcher.megabytes:,.0f} MB -> {_probe_batcher.device} 상주, "
      f"배치 {len(_probe_batcher)}개")
_probe_batcher.release(); del _probe_batcher; gc.collect()

입력 배열 52 MB -> cuda 상주, 배치 151개


0

## 6. 모델

P7 과 구조는 같고 **입력 두 가지가 늘었다.**

- `gather_idx` — 샘플별 gather 인덱스. 모델 버퍼가 아니라 배치로 들어온다 ([R2])
- `flags` — 관측창 이탈 플래그 ([R2])

head 는 12 horizon 에 **가중치를 공유**한다. horizon 을 구분하는 것은
embedding · 탄도창 · gather · 플래그다.

In [6]:
class SolarWindP9(nn.Module):
    def __init__(self, stats):
        super().__init__()
        self.wind_gru = nn.GRU(3, 96, num_layers=2, batch_first=True)
        self.stats_encoder = nn.Sequential(
            nn.Linear(NUM_STATS, 128), nn.SELU(inplace=True),
            nn.Linear(128, 64), nn.SELU(inplace=True))
        shared_dim = 96 + 64

        self.ch_gru = nn.GRU(CH_SEQ_DIM, CH_HIDDEN, num_layers=2, batch_first=True,
                             bidirectional=CH_BIDIRECTIONAL)
        directions = 2 if CH_BIDIRECTIONAL else 1
        self.ch_dropout = nn.Dropout(DROPOUT)
        shared_dim += CH_HIDDEN * directions

        head_extra = 0
        if USE_CH_GATHER:
            self.gather_project = nn.Sequential(
                nn.Linear(CH_HIDDEN * directions, GATHER_DIM), nn.ReLU(inplace=True))
            head_extra += GATHER_DIM * N_GATHER
        if USE_BALLISTIC_WINDOW:
            head_extra += BALLISTIC_DIM
        if USE_TRANSIT_FLAGS:
            head_extra += FLAG_DIM

        self.horizon_embedding = nn.Parameter(torch.randn(12, HORIZON_EMBED) * 0.1)
        head_input = shared_dim + HORIZON_EMBED + head_extra
        self.head = nn.Sequential(
            nn.Linear(head_input, 192), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),
            nn.Linear(192, 96), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),
            nn.Linear(96, 1))

        self.register_buffer("residual_mean", torch.as_tensor(stats["residual_mean"]))
        self.register_buffer("residual_std", torch.as_tensor(stats["residual_std"]))
        if VERBOSE_MODEL:
            print(f"shared={shared_dim} head_input={head_input} "
                  f"(gather {GATHER_DIM * N_GATHER if USE_CH_GATHER else 0}, "
                  f"ballistic {BALLISTIC_DIM if USE_BALLISTIC_WINDOW else 0}, "
                  f"flags {FLAG_DIM if USE_TRANSIT_FLAGS else 0})")

    @staticmethod
    def _gather(projected, index):
        """projected (B, 20, G) 를 샘플별 실수 인덱스 index (B, H, S) 에서 선형보간."""
        batch, horizon, offsets = index.shape
        width = projected.shape[-1]
        lower = index.floor().clamp(0, LAST_INDEX)
        weight = (index - lower).unsqueeze(-1)
        lower = lower.long()
        upper = (lower + 1).clamp(max=int(LAST_INDEX))
        flat_lower = lower.reshape(batch, horizon * offsets, 1).expand(-1, -1, width)
        flat_upper = upper.reshape(batch, horizon * offsets, 1).expand(-1, -1, width)
        low = torch.gather(projected, 1, flat_lower).reshape(batch, horizon, offsets, width)
        high = torch.gather(projected, 1, flat_upper).reshape(batch, horizon, offsets, width)
        return low * (1.0 - weight) + high * weight

    def forward(self, wind_seq, wind_stats, ch_seq, ballistic, gather_idx, flags):
        _, wind_hidden = self.wind_gru(wind_seq)
        ch_sequence, ch_hidden = self.ch_gru(ch_seq)
        if CH_BIDIRECTIONAL:
            ch_last = torch.cat([ch_hidden[-2], ch_hidden[-1]], dim=1)
        else:
            ch_last = ch_hidden[-1]
        shared = torch.cat([F.relu(wind_hidden[-1]), self.stats_encoder(wind_stats),
                            self.ch_dropout(F.relu(ch_last))], dim=1)

        batch = shared.shape[0]
        head_parts = [shared.unsqueeze(1).expand(batch, 12, shared.shape[1]),
                      self.horizon_embedding.unsqueeze(0).expand(batch, 12, HORIZON_EMBED)]
        if USE_CH_GATHER:
            projected = self.gather_project(ch_sequence)
            head_parts.append(self._gather(projected, gather_idx).flatten(2))
        if USE_BALLISTIC_WINDOW:
            head_parts.append(ballistic)
        if USE_TRANSIT_FLAGS:
            head_parts.append(flags)
        z = self.head(torch.cat(head_parts, dim=2)).squeeze(-1)
        return z * self.residual_std + self.residual_mean


def build_model(stats):
    return SolarWindP9(stats).to(DEVICE)


_probe = build_model(_probe_stats)
print("trainable parameters:",
      f"{sum(p.numel() for p in _probe.parameters() if p.requires_grad):,}")
del _probe; gc.collect()

shared=288 head_input=468 (gather 96, ballistic 72, flags 4)
trainable parameters: 312,081


0

## 7. 지표 · 손실 · 기준선

평가지표는 horizon 별 RMSE 를 낸 뒤 평균한 값이다. 손실도 같은 형태로 맞춘다.

기준선을 **두 개** 둔다.

- **persistence** — 마지막 관측값 유지. 단기 horizon 의 하한선
- **climatology** — train 타깃 평균. RMSE 의 사실상 상한선

`skill = 1 - RMSE/climatology_RMSE` 가 0 근처면 모델이 평균만 뱉고 있다는 뜻이고,
그 horizon 은 **모델이 아니라 물리적 예측 한계**에 걸린 것이다. 사다리를 더 올려도 소용없다.

In [7]:
def official_rmse(y_true, y_pred):
    per_horizon = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    return float(per_horizon.mean()), per_horizon


def metric_loss(prediction, target):
    error = (prediction - target) / LOSS_SCALE
    return torch.sqrt((error ** 2).mean(dim=0) + LOSS_EPSILON).mean()


@torch.no_grad()
def predict_with(model, batcher, clip_low, clip_high):
    model.eval()
    predictions = []
    for batch in batcher:
        moved = {k: batch[k].to(DEVICE, non_blocking=PIN_MEMORY) for k in BATCH_KEYS}
        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
            residual = model(moved["wind_seq"], moved["wind_stats"], moved["ch_seq"],
                             moved["ballistic"], moved["gather_idx"], moved["flags"])
        prediction = (residual.float() + moved["last_wind"].unsqueeze(1)
                      ).clamp(clip_low, clip_high)
        predictions.append(prediction.cpu().numpy())
    return np.concatenate(predictions).astype(np.float64), list(batcher.sample_ids)


TRAIN_CLIMATOLOGY = train_targets.mean(axis=0)


def baselines(targets, wind):
    persistence = np.repeat(wind[:, -1:], 12, axis=1)
    climatology = np.tile(TRAIN_CLIMATOLOGY, (len(targets), 1))
    return (np.sqrt(((persistence - targets) ** 2).mean(axis=0)),
            np.sqrt(((climatology - targets) ** 2).mean(axis=0)))


VAL_PERSISTENCE, VAL_CLIMATOLOGY = baselines(val_targets, val_wind)
print(f"validation  persistence {VAL_PERSISTENCE.mean():.3f} / "
      f"climatology {VAL_CLIMATOLOGY.mean():.3f} km/s")
print(pd.DataFrame({"horizon": HORIZONS,
                    "persistence": VAL_PERSISTENCE.round(2),
                    "climatology": VAL_CLIMATOLOGY.round(2)}).to_string(index=False))

validation  persistence 84.105 / climatology 95.275 km/s
 horizon  persistence  climatology
       6    30.129999    94.190002
      12    49.610001    94.540001
      18    63.040001    94.769997
      24    73.260002    95.089996
      30    82.029999    95.199997
      36    89.150002    95.449997
      42    94.699997    95.570000
      48    99.059998    95.639999
      54   102.730003    95.669998
      60   105.949997    95.680000
      66   108.660004    95.709999
      72   110.930000    95.790001


## 8. CV 계측기 — **[C] 다중 시드 · [D] 시간 블록 폴드 · [B] 고정 epoch 비교**

### [D] 폴드 구성

P7 은 사슬을 **샘플 수 기준으로 균등 배치**했다. 사슬 내부 누수는 없지만 시간적으로 인접한
(27일 재현성이 겹치는) 사슬이 서로 다른 폴드에 들어간다. test 는 **미래 계절**이므로
시간 연속 블록이 실제 일반화 격차를 훨씬 잘 반영한다.

| 모드 | 설명 |
|---|---|
| `block` (기본) | 사슬을 시간 순으로 5등분, 한 블록을 홀드아웃 (학습에 나머지 전부) |
| `forward` | 전진 검증 — 블록 $k$ 평가에 블록 $0..k-1$ 만 학습. 가장 정직하지만 표본이 준다 |
| `balanced` | P7 방식. 비교용으로 남겨둠 |

### [C] 다중 시드

`fold_se` 는 **폴드 분산만** 담는다. 이 규모 모델에서 시드 분산은 폴드 분산과 비슷하거나 크다.
사다리 판정은 **시드 3개 × 폴드**의 결합 표준오차로 한다.

### [B] epoch 선택

config 마다 자기 곡선의 argmin 을 쓰면 **선택이 두 번**(epoch, config) 들어가 점수가 낙관적이 된다.
곡선을 평활한 뒤 **기준 rung(A0) 에서 epoch 하나를 정하고**, 전 config 을 그 epoch 에서 비교한다.
각 config 자기 argmin 은 `own_best` 열에 진단용으로만 표시한다 (비교에 쓰지 말 것).

> P11 에서는 이 계측기를 **돌리지 않는다.** [L5] 가 쓰는 `FOLDS` 와 학습 루프 정의만 가져간다.

In [8]:
def build_folds(mode=None, n_folds=N_FOLDS):
    mode = mode or FOLD_MODE
    n_chains = len(TRAIN_CHAINS)
    all_rows = np.arange(len(train_inputs))

    if mode == "balanced":                      # P7 방식 (비교용)
        counts = np.bincount(TRAIN_CHAIN_ID, minlength=n_chains)
        order = np.argsort(counts)[::-1]
        assignment = np.zeros(n_chains, np.int64)
        loads = np.zeros(n_folds, np.int64)
        for chain in order:
            fold = int(np.argmin(loads))
            assignment[chain] = fold
            loads[fold] += counts[chain]
        return [(np.flatnonzero(assignment[TRAIN_CHAIN_ID] != f),
                 np.flatnonzero(assignment[TRAIN_CHAIN_ID] == f)) for f in range(n_folds)]

    edges = np.linspace(0, n_chains, n_folds + 1).astype(int)
    folds = []
    for f in range(n_folds):
        block = np.arange(edges[f], edges[f + 1])
        evaluate = np.flatnonzero(np.isin(TRAIN_CHAIN_ID, block))
        if mode == "forward":
            train = np.flatnonzero(np.isin(TRAIN_CHAIN_ID, np.arange(0, edges[f])))
        else:
            train = np.setdiff1d(all_rows, evaluate)
        if len(train) == 0 or len(evaluate) == 0:
            continue
        folds.append((train, evaluate))
    return folds


FOLDS = build_folds()
print(f"[D] FOLD_MODE={FOLD_MODE} / 폴드 {len(FOLDS)}개")
print("    (학습, 평가) 샘플 수:", [(len(a), len(b)) for a, b in FOLDS])
print("    참고 balanced:", [(len(a), len(b)) for a, b in build_folds('balanced')])


def train_run(train_rows, evaluate_rows, seed=SEED, epochs=CV_EPOCHS, verbose=False):
    """고정 epoch 학습. 매 epoch 의 홀드아웃 horizon별 RMSE 를 기록만 한다."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    stats = fit_stats(train_rows)
    train_loader = make_batcher(
        train_inputs.iloc[train_rows], train_index_matrix[train_rows], train_wind[train_rows],
        train_wind_valid[train_rows], train_grid, stats, train_targets[train_rows],
        shuffle=True, seed=seed, training=True)
    evaluate_loader = make_batcher(
        train_inputs.iloc[evaluate_rows], train_index_matrix[evaluate_rows],
        train_wind[evaluate_rows], train_wind_valid[evaluate_rows], train_grid, stats,
        train_targets[evaluate_rows], shuffle=False, seed=seed)
    evaluate_targets = train_targets[evaluate_rows]

    model = build_model(stats)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

    curve = np.zeros((epochs, 12), np.float64)
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            moved = {k: batch[k].to(DEVICE, non_blocking=PIN_MEMORY)
                     for k in BATCH_KEYS + ("target",)}
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
                residual = model(moved["wind_seq"], moved["wind_stats"], moved["ch_seq"],
                                 moved["ballistic"], moved["gather_idx"], moved["flags"])
            loss = metric_loss(residual.float() + moved["last_wind"].unsqueeze(1),
                               moved["target"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
        scheduler.step()
        prediction, _ = predict_with(model, evaluate_loader,
                                     stats["clip_low"], stats["clip_high"])
        curve[epoch] = official_rmse(evaluate_targets, prediction)[1]
        if verbose:
            print(f"    epoch {epoch + 1:03d} rmse {curve[epoch].mean():7.3f} "
                  f"(72h {curve[epoch][-1]:6.2f})", flush=True)
    train_loader.release(); evaluate_loader.release()
    del model, train_loader, evaluate_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return curve


def smooth_curve(curve, window=EPOCH_SMOOTH):
    """[B] epoch 축 이동평균. argmin 이 단발 노이즈를 집는 것을 막는다."""
    if window <= 1:
        return curve
    left = window // 2
    padded = np.pad(curve, ((left, window - 1 - left), (0, 0)), mode="edge")
    return np.stack([padded[i:i + window].mean(axis=0) for i in range(len(curve))])


def settings_signature(settings):
    """설정 해시. 캐시 키에 넣어야 설정을 바꿨을 때 옛 결과를 조용히 재사용하지 않는다."""
    payload = json.dumps({k: str(v) for k, v in sorted(settings.items())})
    payload += f"|{FOLD_MODE}"
    return hashlib.md5(payload.encode()).hexdigest()[:8]


def run_cv(label, folds, seeds=CV_SEEDS, epochs=CV_EPOCHS, resume=True, signature=""):
    """-> curves (n_folds, n_seeds, epochs, 12). 칸 단위로 저장하고 재시작 시 건너뛴다."""
    path = OUTPUT_DIR / f"cv_{label}{'_' + signature if signature else ''}.npy"
    shape = (len(folds), len(seeds), epochs, 12)
    if resume and path.exists():
        cached = np.load(path)
        if cached.shape == shape:
            print(f"  [{label}] 캐시 재사용 {path.name} "
                  f"(다시 돌리려면 파일을 지우거나 resume=False)", flush=True)
            return cached
        print(f"  [{label}] 캐시 형상 불일치 {cached.shape} != {shape} -> 재계산")

    started = time.perf_counter()
    curves = np.zeros(shape)
    total = len(folds) * len(seeds)
    for f, (train_rows, evaluate_rows) in enumerate(folds):
        for s, seed in enumerate(seeds):
            curves[f, s] = train_run(train_rows, evaluate_rows, seed, epochs)
            done = f * len(seeds) + s + 1
            elapsed = time.perf_counter() - started
            print(f"  [{label}] fold {f + 1}/{len(folds)} seed {seed} "
                  f"best {curves[f, s].mean(axis=1).min():7.3f} | "
                  f"{done}/{total} runs, {elapsed / 60:.1f}분 경과, "
                  f"남은 {elapsed / done * (total - done) / 60:.1f}분", flush=True)
    np.save(path, curves)
    print(f"  [{label}] 완료 ({(time.perf_counter() - started) / 60:.1f}분) -> {path.name}",
          flush=True)
    return curves


def paired_scores(curves, epoch):
    """(fold, seed) 조합별 평균 RMSE. -> (F*S,), (F*S, 12)"""
    smoothed = np.stack([[smooth_curve(curves[f, s]) for s in range(curves.shape[1])]
                         for f in range(curves.shape[0])])
    per_horizon = smoothed[:, :, epoch].reshape(-1, 12)
    return per_horizon.mean(axis=1), per_horizon


def paired_delta(baseline, variant, epoch, columns=slice(None)):
    """**같은 (fold, seed) 쌍**에서의 차이를 본다.

    두 config 은 폴드와 시드를 공유한다. 독립 표본처럼 비교하면 공통 분산(그 폴드가
    원래 어려웠다, 그 시드가 원래 나빴다)이 오차에 그대로 남는다. 짝을 지어 빼면
    그 성분이 상쇄되고, 같은 판정력을 훨씬 적은 실행으로 얻는다.
    """
    _, base_h = paired_scores(baseline, epoch)
    _, var_h = paired_scores(variant, epoch)
    difference = var_h[:, columns].mean(axis=1) - base_h[:, columns].mean(axis=1)
    n = len(difference)
    se = float(difference.std(ddof=1) / np.sqrt(n)) if n > 1 else np.nan
    return {"delta": float(difference.mean()), "se": se, "n": n,
            "per_pair": difference,
            "unpaired_se": float(np.hypot(base_h[:, columns].mean(axis=1).std(ddof=1),
                                          var_h[:, columns].mean(axis=1).std(ddof=1))
                                 / np.sqrt(n)) if n > 1 else np.nan}


def verdict(delta, se, sigma=NOISE_SIGMA):
    if not np.isfinite(se):
        return "판정불가"
    if delta + sigma * se < 0:
        return "개선"
    if delta - sigma * se > 0:
        return "악화"
    return "차이없음"


def score_at(curves, epoch):
    """(fold, seed) 별 평균 RMSE 와 horizon별 평균. [C] 결합 표준오차."""
    smoothed = np.stack([[smooth_curve(curves[f, s]) for s in range(curves.shape[1])]
                         for f in range(curves.shape[0])])          # (F, S, E, 12)
    per_horizon = smoothed[:, :, epoch].reshape(-1, 12)             # (F*S, 12)
    runs = per_horizon.mean(axis=1)
    fold_means = smoothed[:, :, epoch].mean(axis=(1, 2))            # 폴드별
    seed_means = smoothed[:, :, epoch].mean(axis=(0, 2))            # 시드별
    return {"per_horizon": per_horizon.mean(axis=0),
            "cv": float(runs.mean()),
            "se": float(runs.std(ddof=1) / np.sqrt(len(runs))) if len(runs) > 1 else np.nan,
            "fold_sd": float(fold_means.std(ddof=1)) if len(fold_means) > 1 else np.nan,
            "seed_sd": float(seed_means.std(ddof=1)) if len(seed_means) > 1 else np.nan,
            "smoothed_mean": smoothed.mean(axis=(0, 1))}

[D] FOLD_MODE=block / 폴드 5개
    (학습, 평가) 샘플 수: [(7072, 2535), (7140, 2467), (8498, 1109), (8110, 1497), (7608, 1999)]
    참고 balanced: [(7661, 1946), (7696, 1911), (7677, 1930), (7695, 1912), (7699, 1908)]


## P11 — 물리 피처 확장 + 무적합 앙상블

P9 셀 0~16 은 그대로 두고 여기서부터 함수를 재정의한다.
**제출할 때 바꾸는 것은 아래 셀의 `STAGE` 한 줄뿐이다.**

| STAGE | 내용 |
|---|---|
| S1 | 앙상블만 (구 기하) — 바닥 확보 |
| S2 | + [L2] μ 보정 · 등각 경도 |
| S3 | + [L3] θ_b · 형태 피처 |
| S4 | + [L4] 탄도창 확장 + [L5] 수축 보정 |
| S5 | 최종 — 시드 5개로 확대 (멤버 15) |
| S5M | 최종 대안 — 구 기하 + 새 기하를 같이 평균 (멤버 12) |

In [9]:
# ============================  P11 레이어 시작  ============================
# 위 셀(P9 원본)은 한 글자도 고치지 않았다. 아래에서 함수를 재정의해 덮어쓴다.
# P9 경로로 되돌리려면 이 셀부터 끝까지 실행하지 않으면 된다.

P11_VERSION = "p11a"

# ---- [L2] 기하 보정 --------------------------------------------------------
AREA_SOURCE     = "mu_approx"   # "raw" | "mu_approx"(재추출 불필요) | "true"(p11a 필요)
EQUAL_ANGLE_LON = True          # 경도 셀을 등각으로 (x 등간격 -> sinφ 등간격)
MU_FLOOR        = 0.30          # 1/μ 상한 클램프 (최대 3.33배). 잘라내지는 않는다

# ---- [L3] 물리 피처 (p11a 캐시 필요) ---------------------------------------
USE_DEPTH          = False      # 셀별 θ_b (최대 · 총합) — WSA 경계거리 항
USE_RATIO          = False      # 셀별 211/193 비 (온도 대리값)
USE_FRAME          = False      # 프레임 형태 스칼라
DEPTH_IN_BALLISTIC = False      # 탄도 소스 시각에서도 θ_b 를 뽑는다
FRAME_USE = ["big_frac_true", "n_components", "big_lat_deg", "big_lon_deg", "big_depth_deg"]

# ---- [L5] 수축 보정 --------------------------------------------------------
FIT_SHRINKAGE = False           # train OOF 로 horizon별 λ 적합 (폴드 수만큼 학습 추가)
SHRINK_CLIP   = (0.70, 1.00)
LAM_H = np.ones(12, np.float64)

# ---- 앙상블 ----------------------------------------------------------------
STAGE = "S1"                    # S1->S5 / S5M. **제출할 때 여기만 바꾼다** (아래 표 참고)
ENSEMBLE_SEEDS = (777, 778, 779)
ENSEMBLE_SEEDS_FINAL = (777, 778, 779, 780, 781)

# ---- p11a 캐시 적재 --------------------------------------------------------
P11_FRAME_COLUMNS = ["cy", "cx", "radius", "total_px", "total_true",
                     "big_frac", "big_frac_true", "n_components",
                     "big_lat_deg", "big_lon_deg", "big_depth_deg"]
P11_KEYS = ("area", "area_true", "depth_max", "depth_sum", "ratio", "frame")


def load_p11(split, files):
    path = CACHE_ROOT / f"{P11_VERSION}_{split}.npz"
    if not path.exists():
        return None
    with np.load(path) as blob:
        if [str(n) for n in blob["files"]] != list(files):
            print(f"  ⚠️ {path.name}: 파일 목록이 다르다 -> 쓰지 않는다")
            return None
        return {k: np.asarray(blob[k], np.float32) for k in P11_KEYS}


P11 = {}
for _split, _files in (("train", train_files), ("validation", val_files), ("test", test_files)):
    _loaded = load_p11(_split, _files)
    if _loaded is not None:
        P11[_split] = _loaded
P11_AVAILABLE = len(P11) == 3

print(f"p11a 캐시: {'사용 가능' if P11_AVAILABLE else '없음'}")
if P11_AVAILABLE:
    # p11a 의 `area` 는 p7a 와 **같은 알고리즘·같은 파일 순서**로 낸 것이다.
    # 어긋나면 area_true·depth 도 다른 프레임을 가리키고 있다는 뜻이라 여기서 멈춘다.
    _gap = float(np.abs(P11["train"]["area"] - train_area).max())
    print(f"  p7a 대조: 면적 최대 오차 {_gap:.2e} (0 이어야 정상)")
    assert _gap < 1e-5, "p11a 가 p7a 와 다르다 — 파일 순서 / 원반 검출을 확인할 것"
    del _gap
    # 형태 피처는 cv2/scipy 가 있어야 나온다. 없으면 상수 열이 되므로 미리 뺀다.
    _frame = P11["train"]["frame"]
    _live = [c for c in FRAME_USE
             if np.isfinite(_frame[:, P11_FRAME_COLUMNS.index(c)]).any()
             and float(np.nanstd(_frame[:, P11_FRAME_COLUMNS.index(c)])) > 1e-6]
    if _live != FRAME_USE:
        print(f"  상수/결측 프레임 스칼라 제외: {[c for c in FRAME_USE if c not in _live]}")
        FRAME_USE = _live
    del _frame, _live
else:
    print("  -> [L3] 물리 피처를 끄고 [L2] 근사판만 쓴다. p11_extract.py 를 먼저 돌릴 것.")
    if AREA_SOURCE == "true":
        AREA_SOURCE = "mu_approx"
    USE_DEPTH = USE_RATIO = USE_FRAME = DEPTH_IN_BALLISTIC = False

p11a 캐시: 없음
  -> [L3] 물리 피처를 끄고 [L2] 근사판만 쓴다. p11_extract.py 를 먼저 돌릴 것.


### P11 피처 레이어 — `aggregate` · `configure` · `flatten_ch` 재정의

In [10]:
from collections import namedtuple

P11Grid = namedtuple("P11Grid", "seq cells")


def mu_weights(fine_lat=FINE_LAT, fine_lon=FINE_LON, mu_floor=None, side=512):
    """fine 셀별 평균 1/μ. 캐시된 픽셀면적에 곱하면 근사 진짜면적이 된다.

    fine 격자는 반지름 r_used = R * DISK_MARGIN 의 외접 사각형을 등분한 것이므로,
    μ 는 **진짜 반지름 R** 기준으로 되돌려서 재야 한다.
    """
    mu_floor = MU_FLOOR if mu_floor is None else mu_floor
    grid = np.linspace(-1.0, 1.0, side)
    Y, X = np.meshgrid(grid, grid, indexing="ij")
    rho2 = X ** 2 + Y ** 2
    on_disk = rho2 <= 1.0
    mu = np.sqrt(np.clip(1.0 - rho2 * DISK_MARGIN ** 2, 0.0, 1.0))
    weight = 1.0 / np.clip(mu, mu_floor, 1.0)
    li = np.clip(((Y + 1) / 2 * fine_lat).astype(int), 0, fine_lat - 1)
    lj = np.clip(((X + 1) / 2 * fine_lon).astype(int), 0, fine_lon - 1)
    cell = li * fine_lon + lj
    numerator = np.bincount(cell[on_disk], weights=weight[on_disk],
                            minlength=fine_lat * fine_lon)
    denominator = np.bincount(cell[on_disk], minlength=fine_lat * fine_lon)
    return (numerator / np.maximum(denominator, 1)).astype(np.float32)


MU_W = mu_weights()


def lon_columns(fine_lon, grid_lon, equal_angle):
    """fine 경도 열 -> coarse 열 배정. 등각이면 sinφ 경계로 나눈다."""
    if not equal_angle:
        return np.minimum((np.arange(fine_lon) * grid_lon) // fine_lon, grid_lon - 1)
    centers = ((np.arange(fine_lon) + 0.5) / fine_lon) * 2.0 - 1.0      # x / r_used
    phi = np.degrees(np.arcsin(np.clip(centers * DISK_MARGIN, -1.0, 1.0)))
    return np.clip(((phi + 90.0) / 180.0 * grid_lon).astype(int), 0, grid_lon - 1)


def _reduce(block, axis, how):
    if how == "sum":
        return block.sum(axis)
    if how == "mean":
        return block.mean(axis)
    return block.max(axis)


def aggregate_fine(fine, grid_lat, grid_lon, fold, lon_map, how="sum"):
    """fine (n, FINE_CELLS) -> (n, USED_LAT * grid_lon).

    위도는 P9 그대로 균등 블록이다 (y 등간격 = sinθ 등간격 = **등면적 밴드**).
    경도만 lon_map 으로 묶는다. 접기는 P9 와 같은 순서·같은 방향이다.
    """
    n = len(fine)
    block = fine.reshape(n, FINE_LAT, FINE_LON)
    block = block.reshape(n, grid_lat, FINE_LAT // grid_lat, FINE_LON)
    block = _reduce(block, 2, how)
    out = np.empty((n, grid_lat, grid_lon), np.float32)
    for column in range(grid_lon):
        out[:, :, column] = _reduce(block[:, :, lon_map == column], 2, how)
    if fold:
        flipped = out[:, ::-1, :]
        if how == "max":
            out = np.maximum(out, flipped)
        elif how == "mean":
            out = (out + flipped) / 2.0
        else:
            out = out + flipped
        out = out[:, : (grid_lat + 1) // 2, :]
    return np.ascontiguousarray(out.reshape(n, -1), dtype=np.float32)


def channel_fine(split, area_raw, name):
    """채널 이름 -> (fine 격자 (n, FINE_CELLS), 집계 방식)."""
    if name in LEVEL_NAMES:
        level = LEVEL_NAMES.index(name)
        if AREA_SOURCE == "true" and P11_AVAILABLE:
            return P11[split]["area_true"][:, level], "sum"
        fine = area_raw[:, level]
        if AREA_SOURCE == "mu_approx":
            fine = fine * MU_W
        return fine, "sum"
    if name == "depth_max":
        return P11[split]["depth_max"], "max"        # 깊이는 더하면 안 된다
    if name == "depth_mass":
        return P11[split]["depth_sum"], "sum"        # Σθ_b — 크기까지 반영한 양
    if name == "ratio":
        return P11[split]["ratio"], "mean"
    raise KeyError(name)


def frame_features(split):
    if not (USE_FRAME and P11_AVAILABLE and FRAME_USE):
        return None
    columns = [P11_FRAME_COLUMNS.index(c) for c in FRAME_USE]
    return np.nan_to_num(P11[split]["frame"][:, columns], nan=0.0).astype(np.float32)


def build_grid(split, area_raw):
    """-> P11Grid(seq=(n, D) 프레임 피처, cells=(n, Cb, N_CELLS) 탄도용)."""
    stacked, ballistic = [], []
    for name in CH_CHANNELS:
        fine, how = channel_fine(split, area_raw, name)
        coarse = aggregate_fine(fine, GRID_LAT, GRID_LON, FOLD_LATITUDE, LON_MAP, how)
        stacked.append(coarse)
        if name in BALLISTIC_CHANNELS:
            ballistic.append(coarse)
    stacked = np.stack(stacked, axis=1)
    sequence = stacked.reshape(len(stacked), -1)
    frame = frame_features(split)
    if frame is not None:
        sequence = np.concatenate([sequence, frame], axis=1)
    return P11Grid(seq=np.ascontiguousarray(sequence, np.float32),
                   cells=np.ascontiguousarray(np.stack(ballistic, axis=1), np.float32))


def configure(**overrides):
    """P9 의 configure 를 대체한다. ablation·멤버 전환은 이 함수로만 한다."""
    globals().update(overrides)
    global GRID_LAT, GRID_LON, USED_LAT, N_CELLS, CENTRAL_LON, EQUATOR_ROW
    global LEVEL_INDEX, N_USED_LEVELS, CH_SEQ_DIM, CH_CHANNELS, BALLISTIC_CHANNELS
    global train_grid, val_grid, test_grid, LON_MAP
    global AREA_LAT_ROWS, N_AREA_OFFSETS, BALLISTIC_DIM, N_GATHER

    GRID_LAT, GRID_LON = CH_GRID
    assert FINE_LAT % GRID_LAT == 0, "위도 격자가 fine 격자를 나누지 못한다"
    USED_LAT = (GRID_LAT + 1) // 2 if FOLD_LATITUDE else GRID_LAT
    N_CELLS = USED_LAT * GRID_LON
    CENTRAL_LON = GRID_LON // 2
    EQUATOR_ROW = USED_LAT - 1 if FOLD_LATITUDE else GRID_LAT // 2
    LEVEL_INDEX = [LEVEL_NAMES.index(name) for name in USE_LEVELS]
    N_USED_LEVELS = len(LEVEL_INDEX)

    LON_MAP = lon_columns(FINE_LON, GRID_LON, EQUAL_ANGLE_LON)
    assert len(np.unique(LON_MAP)) == GRID_LON, "빈 경도 열이 생겼다 — GRID_LON 을 줄일 것"

    CH_CHANNELS = list(USE_LEVELS)
    if USE_DEPTH and P11_AVAILABLE:
        CH_CHANNELS += ["depth_max", "depth_mass"]
    if USE_RATIO and P11_AVAILABLE:
        CH_CHANNELS += ["ratio"]
    BALLISTIC_CHANNELS = list(USE_LEVELS)
    if DEPTH_IN_BALLISTIC and USE_DEPTH and P11_AVAILABLE:
        BALLISTIC_CHANNELS = BALLISTIC_CHANNELS + ["depth_max"]

    train_grid = build_grid("train", train_area)
    val_grid = build_grid("validation", val_area)
    test_grid = build_grid("test", test_area)

    CH_SEQ_DIM = int(train_grid.seq.shape[1])
    AREA_LAT_ROWS = list(range(USED_LAT)) if BALLISTIC_LAT == "profile" else [EQUATOR_ROW]
    N_AREA_OFFSETS = (len(BALLISTIC_OFFSETS) if BALLISTIC_SOURCE == "window"
                      else len(TRANSIT_SPEEDS))
    _, n_lons = ballistic_columns()
    BALLISTIC_DIM = (N_AREA_OFFSETS * len(AREA_LAT_ROWS) * n_lons
                     * len(BALLISTIC_CHANNELS))
    N_GATHER = len(GATHER_OFFSETS) if ADAPTIVE_GATHER else len(TRANSIT_SPEEDS)


def flatten_ch(grid, indexes):
    """(n, 20) 인덱스 -> (n, 20, CH_SEQ_DIM)."""
    return grid.seq[indexes].astype(np.float32)


def ballistic_area(grid, indexes, index):
    """탄도 소스 시각의 셀 값. index (n, 12, K) -> (n, 12, K*D)."""
    columns, _ = ballistic_columns()
    sequence = grid.cells[indexes][:, :, :, columns]
    sequence = sequence.reshape(len(indexes), 20, -1)
    n_samples, n_horizon, n_offset = index.shape
    picked = pick_per_sample(sequence, index.reshape(n_samples, n_horizon * n_offset))
    return picked.reshape(n_samples, n_horizon, -1).astype(np.float32)


configure()
print(f"P11 격자 {GRID_LAT}x{GRID_LON} -> 셀 {N_CELLS} | 채널 {CH_CHANNELS}")
print(f"  ch_seq {CH_SEQ_DIM} / 탄도 {BALLISTIC_DIM} (채널 {BALLISTIC_CHANNELS})")
print(f"  면적 소스 {AREA_SOURCE} / 등각 경도 {EQUAL_ANGLE_LON} / LON_MAP {LON_MAP.tolist()}")

# --- 기하 보정이 실제로 무엇을 바꿨는지 한 번 본다 --------------------------
_raw = aggregate_fine(train_area[:, LEVEL_INDEX[0]], GRID_LAT, GRID_LON, FOLD_LATITUDE,
                      lon_columns(FINE_LON, GRID_LON, False), "sum")
_fix = aggregate_fine(train_area[:, LEVEL_INDEX[0]] * MU_W, GRID_LAT, GRID_LON,
                      FOLD_LATITUDE, lon_columns(FINE_LON, GRID_LON, True), "sum")
print(f"\n[L2] 셀별 총 CH 면적 원본 {_raw.sum(1).mean():.4f} -> 보정 {_fix.sum(1).mean():.4f}")
print(f"     프레임간 상대변동(std/mean) {_raw.sum(1).std() / _raw.sum(1).mean():.4f} -> "
      f"{_fix.sum(1).std() / _fix.sum(1).mean():.4f}  (작을수록 가짜 변조가 줄었다는 뜻)")
del _raw, _fix

P11 격자 6x3 -> 셀 9 | 채널 ['dark0.45', 'bright']
  ch_seq 18 / 탄도 72 (채널 ['dark0.45', 'bright'])
  면적 소스 mu_approx / 등각 경도 True / LON_MAP [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2]

[L2] 셀별 총 CH 면적 원본 0.0370 -> 보정 0.0592
     프레임간 상대변동(std/mean) 0.5778 -> 0.5382  (작을수록 가짜 변조가 줄었다는 뜻)


### 멤버 구성 — 단계별로 어떤 모델들을 평균할 것인가

In [11]:
# ---- P9 사다리 설정 (셀 18 에서 그대로 옮겨 왔다) --------------------------
ABLATION_LADDER = [
    ("A0. P3 재현", dict(
        CH_GRID=(3, 5), FOLD_LATITUDE=False, USE_LEVELS=("dark0.45",),
        TRANSIT_SPEEDS=(350.0, 500.0, 700.0), BALLISTIC_SOURCE="speeds",
        BALLISTIC_LAT="equator", BALLISTIC_LON="central", BALLISTIC_OFFSETS=(0,),
        ADAPTIVE_AREA=False, ADAPTIVE_GATHER=False, USE_TRANSIT_FLAGS=False,
        USE_CH_GATHER=False, USE_BALLISTIC_WINDOW=True, CH_BIDIRECTIONAL=False)),
    ("B0. + 탄도속도 재설정", dict(TRANSIT_SPEEDS=(315.0, 385.0, 500.0))),
    ("C0. + 격자축·적도대칭", dict(CH_GRID=(6, 3), FOLD_LATITUDE=True)),
    ("D0. + 활성영역 레벨", dict(USE_LEVELS=("dark0.45", "bright"))),
    ("E0. + 탄도창(시각 4점)", dict(BALLISTIC_SOURCE="window",
                                    BALLISTIC_OFFSETS=(-4, -2, 0, 2))),
    ("E1. + 위도 프로파일", dict(BALLISTIC_LAT="profile")),
    ("E2. + 경도 전체 [R1]", dict(BALLISTIC_LON="all")),
    ("F1. + gather(단방향)", dict(USE_CH_GATHER=True)),
    ("F2. + 양방향 GRU", dict(CH_BIDIRECTIONAL=True)),
    ("F3. + gather 속도 확장", dict(TRANSIT_SPEEDS=(315.0, 345.0, 385.0,
                                                    435.0, 500.0, 600.0))),
]


def config_at(code):
    settings = {}
    for label, overrides in ABLATION_LADDER:
        settings.update(overrides)
        if label.split(".")[0] == code:
            return dict(settings)
    raise KeyError(code)


# ---- 제출 단계별 멤버 구성 -------------------------------------------------
GEOMETRY_OFF = dict(AREA_SOURCE="raw", EQUAL_ANGLE_LON=False)
GEOMETRY_ON = dict(AREA_SOURCE="true" if P11_AVAILABLE else "mu_approx",
                   EQUAL_ANGLE_LON=True)
PHYSICS_OFF = dict(USE_DEPTH=False, USE_RATIO=False, USE_FRAME=False,
                   DEPTH_IN_BALLISTIC=False)
PHYSICS_ON = dict(USE_DEPTH=True, USE_RATIO=False, USE_FRAME=True,
                  DEPTH_IN_BALLISTIC=True)
BALLISTIC_WIDE = dict(BALLISTIC_OFFSETS=(-8, -5, -2, 0, 2, 5))

# STAGE 하나가 기하·물리·탄도창·수축·시드 수를 전부 정한다. 따로 켤 스위치는 없다.
#   S1~S5 : 제출 사다리. 한 칸 올릴 때마다 레버가 하나씩 추가된다.
#   S5M   : 계획 §2 의 K=12 구성 — **구 기하 멤버와 새 기하 멤버를 같이 평균한다.**
#           S2 가 "기하 보정이 이득"이라고 말했더라도, 두 기하는 서로 다른 오차를 내므로
#           섞는 편이 평균의 이득이 크다. S5 와 둘 중 하나를 고르면 된다.
STAGE_TABLE = {
    "S1":  dict(geometry=GEOMETRY_OFF, physics=False, wide=False, shrink=False, seeds="normal"),
    "S2":  dict(geometry=GEOMETRY_ON,  physics=False, wide=False, shrink=False, seeds="normal"),
    "S3":  dict(geometry=GEOMETRY_ON,  physics=True,  wide=False, shrink=False, seeds="normal"),
    "S4":  dict(geometry=GEOMETRY_ON,  physics=True,  wide=True,  shrink=True,  seeds="normal"),
    "S5":  dict(geometry=GEOMETRY_ON,  physics=True,  wide=True,  shrink=True,  seeds="final"),
    "S5M": dict(geometry=GEOMETRY_ON,  physics=True,  wide=True,  shrink=True,  seeds="normal",
                mixed=True),
}


def member_groups(spec):
    """(이름표, 사다리 코드, 기하, 물리) 목록. 각 그룹이 시드 수만큼 멤버를 낸다."""
    if spec.get("mixed"):
        return [("A0", "A0", GEOMETRY_OFF, PHYSICS_OFF),
                ("F3", "F3", GEOMETRY_OFF, PHYSICS_OFF),
                ("L2", "F3", GEOMETRY_ON, PHYSICS_OFF),
                ("L3", "F3", GEOMETRY_ON, PHYSICS_ON)]
    groups = [("A0", "A0", spec["geometry"], PHYSICS_OFF),
              ("F3", "F3", spec["geometry"], PHYSICS_OFF)]
    if spec["physics"]:
        groups.append(("L3", "F3", spec["geometry"], PHYSICS_ON))
    return groups


def members_for(stage):
    spec = STAGE_TABLE[stage]
    seeds = ENSEMBLE_SEEDS_FINAL if spec["seeds"] == "final" else ENSEMBLE_SEEDS
    ballistic = BALLISTIC_WIDE if spec["wide"] else {}
    members = []
    for label, code, geometry, physics in member_groups(spec):
        extra = {} if code == "A0" else ballistic      # A0 는 speeds 모드라 창이 없다
        for i, seed in enumerate(seeds):
            members.append(dict(
                name=f"{label}_s{seed}_e{1 + i % 2}",
                settings={**config_at(code), **physics, **geometry, **extra},
                seed=seed, epochs=1 + i % 2))
    return members


STAGE_SPEC = STAGE_TABLE[STAGE]
if STAGE_SPEC["physics"] and not P11_AVAILABLE:
    print("⚠️ p11a 캐시가 없다 -> L3 멤버가 F3 와 같은 설정이 된다 (시드 다양성만 는다).")
FIT_SHRINKAGE = STAGE_SPEC["shrink"]           # [L5] 도 STAGE 가 켠다
MEMBERS = members_for(STAGE)

print(f"STAGE = {STAGE} | 멤버 {len(MEMBERS)}개 | "
      f"면적 {STAGE_SPEC['geometry']['AREA_SOURCE']} · "
      f"등각경도 {STAGE_SPEC['geometry']['EQUAL_ANGLE_LON']} · "
      f"물리 {STAGE_SPEC['physics']} · 탄도창확장 {STAGE_SPEC['wide']} · "
      f"수축 {FIT_SHRINKAGE}"
      + (" · **기하 혼합**" if STAGE_SPEC.get("mixed") else ""))
for m in MEMBERS:
    print(f"  {m['name']:<16s} seed {m['seed']} epoch {m['epochs']}")

STAGE = S1 | 멤버 6개 | 면적 raw · 등각경도 False · 물리 False · 탄도창확장 False · 수축 False
  A0_s777_e1       seed 777 epoch 1
  A0_s778_e2       seed 778 epoch 2
  A0_s779_e1       seed 779 epoch 1
  F3_s777_e1       seed 777 epoch 1
  F3_s778_e2       seed 778 epoch 2
  F3_s779_e1       seed 779 epoch 1


### 멤버 학습 · 무적합 평균

적합 파라미터 0개의 균등가중 평균이다. P6 의 NNLS(val 로 84개 적합)와 다르다.

In [12]:
def train_full(stats, epochs, seed):
    """train 전체로 고정 epoch 학습. LR 궤적은 P9 최종학습과 동일하게 T_max=CV_EPOCHS."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    loader = make_batcher(train_inputs, train_index_matrix, train_wind, train_wind_valid,
                          train_grid, stats, train_targets,
                          shuffle=True, seed=seed, training=True)
    model = build_model(stats)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CV_EPOCHS)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    for _ in range(epochs):
        model.train()
        for batch in loader:
            moved = {k: batch[k].to(DEVICE, non_blocking=PIN_MEMORY)
                     for k in BATCH_KEYS + ("target",)}
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
                residual = model(moved["wind_seq"], moved["wind_stats"], moved["ch_seq"],
                                 moved["ballistic"], moved["gather_idx"], moved["flags"])
            loss = metric_loss(residual.float() + moved["last_wind"].unsqueeze(1),
                               moved["target"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
        scheduler.step()
    loader.release()
    return model


def predict_split(model, inputs, index_matrix, wind, valid, grid, stats, targets=None):
    batcher = make_batcher(inputs, index_matrix, wind, valid, grid, stats, targets,
                           shuffle=False)
    prediction, ids = predict_with(model, batcher, stats["clip_low"], stats["clip_high"])
    batcher.release()
    return prediction, ids


# ============================  멤버 학습  ==================================
VERBOSE_MODEL = False
MEMBER_STATES, MEMBER_STATS, VAL_PREDICTIONS, TEST_PREDICTIONS = [], [], [], []
_started = time.perf_counter()

for index, member in enumerate(MEMBERS):
    configure(**member["settings"])
    stats = fit_stats(np.arange(len(train_inputs)))          # train 행만 — val 미사용
    model = train_full(stats, member["epochs"], member["seed"])

    val_prediction, val_ids = predict_split(model, val_inputs, val_index_matrix, val_wind,
                                            val_wind_valid, val_grid, stats, val_targets)
    test_prediction, test_ids = predict_split(model, test_inputs, test_index_matrix,
                                              test_wind, test_wind_valid, test_grid, stats)
    assert val_ids == val_inputs.sample_id.tolist()
    assert test_ids == test_inputs.sample_id.tolist()

    MEMBER_STATES.append({k: v.detach().cpu() for k, v in model.state_dict().items()})
    MEMBER_STATS.append(stats)
    VAL_PREDICTIONS.append(val_prediction)
    TEST_PREDICTIONS.append(test_prediction)
    score, _ = official_rmse(val_targets, val_prediction)
    print(f"  [{index + 1}/{len(MEMBERS)}] {member['name']:<16s} "
          f"ch_seq {CH_SEQ_DIM:3d} 탄도 {BALLISTIC_DIM:3d} | val {score:7.3f} | "
          f"{(time.perf_counter() - _started) / 60:.1f}분 경과", flush=True)
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
VERBOSE_MODEL = True

VAL_MEAN = np.mean(VAL_PREDICTIONS, axis=0)
TEST_MEAN = np.mean(TEST_PREDICTIONS, axis=0)

# ---- 앙상블이 수학적으로 기대한 대로 동작했는지만 확인한다 -----------------
# (val 로 **모델을 고르지는 않는다**. 평균이 개별 멤버보다 나은 것은 항등식에 가깝고,
#  그게 안 나오면 파이프라인이 깨진 것이므로 이 확인은 선택과 무관하다.)
_member_scores = [official_rmse(val_targets, p)[0] for p in VAL_PREDICTIONS]
_ensemble_score, _ = official_rmse(val_targets, VAL_MEAN)
_pairs = [(a - b) ** 2 for i, a in enumerate(TEST_PREDICTIONS)
          for b in TEST_PREDICTIONS[i + 1:]]
_disagreement = float(np.sqrt(np.mean(_pairs))) if _pairs else 0.0
_mean_member = float(np.mean(_member_scores))
print(f"\n멤버 val 평균 {_mean_member:.3f} (최저 {min(_member_scores):.3f}) "
      f"-> 앙상블 {_ensemble_score:.3f}")
print(f"멤버간 test 예측 불일치 {_disagreement:.2f} RMS km/s "
      f"(클수록 평균의 이득이 크다)")
# 기준은 **최고 멤버**가 아니라 **멤버 평균**이다. 볼록성이 보장하는 것은
# "평균의 오차 <= 오차의 평균" 까지이고, 최고 멤버를 이기는 것은 보장되지 않는다.
# (최고 멤버를 기준으로 두면 val 로 멤버를 고르는 것과 같아진다 — P6 의 함정)
if len(MEMBERS) > 1 and _ensemble_score > _mean_member:
    print(f"🔴 앙상블이 멤버 평균보다 나쁘다 — 예측 정렬/통계/클립을 의심할 것")
assert len(MEMBERS) == 1 or _ensemble_score <= _mean_member + 0.5, \
    "앙상블이 멤버 평균보다 크게 나쁘다 — 파이프라인이 깨졌다"

  [1/6] A0_s777_e1       ch_seq  15 탄도   3 | val  66.608 | 0.1분 경과
  [2/6] A0_s778_e2       ch_seq  15 탄도   3 | val  67.653 | 0.1분 경과
  [3/6] A0_s779_e1       ch_seq  15 탄도   3 | val  66.982 | 0.1분 경과
  [4/6] F3_s777_e1       ch_seq  18 탄도  72 | val  66.996 | 0.2분 경과
  [5/6] F3_s778_e2       ch_seq  18 탄도  72 | val  69.304 | 0.2분 경과
  [6/6] F3_s779_e1       ch_seq  18 탄도  72 | val  67.234 | 0.3분 경과

멤버 val 평균 67.463 (최저 66.608) -> 앙상블 65.622
멤버간 test 예측 불일치 22.53 RMS km/s (클수록 평균의 이득이 크다)


### [L5] 수축 보정 — train OOF 에서만 적합

In [13]:
# ============================  [L5] 수축 보정  ==============================
# RMSE 최적 예측은 조건부 평균이다. 고분산 추정기는 평균 주위로 과하게 흔들리므로
# horizon 별 기후값 쪽으로 수축시키면 RMSE 가 내려간다.
# **적합은 train OOF 에서만 한다** — val 에 12개라도 적합하면 P6 의 함정을 반복한다.

def oof_predictions(settings, epochs, seed):
    """폴드별 홀드아웃 예측을 모아 train 전체 크기로 돌려준다."""
    configure(**settings)
    out = np.full((len(train_inputs), 12), np.nan)
    for train_rows, evaluate_rows in FOLDS:
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        stats = fit_stats(train_rows)
        loader = make_batcher(train_inputs.iloc[train_rows], train_index_matrix[train_rows],
                              train_wind[train_rows], train_wind_valid[train_rows],
                              train_grid, stats, train_targets[train_rows],
                              shuffle=True, seed=seed, training=True)
        model = build_model(stats)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                      weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CV_EPOCHS)
        scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
        for _ in range(epochs):
            model.train()
            for batch in loader:
                moved = {k: batch[k].to(DEVICE, non_blocking=PIN_MEMORY)
                         for k in BATCH_KEYS + ("target",)}
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
                    residual = model(moved["wind_seq"], moved["wind_stats"],
                                     moved["ch_seq"], moved["ballistic"],
                                     moved["gather_idx"], moved["flags"])
                loss = metric_loss(residual.float() + moved["last_wind"].unsqueeze(1),
                                   moved["target"])
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer); scaler.update()
            scheduler.step()
        loader.release()
        prediction, _ = predict_split(model, train_inputs.iloc[evaluate_rows],
                                      train_index_matrix[evaluate_rows],
                                      train_wind[evaluate_rows],
                                      train_wind_valid[evaluate_rows],
                                      train_grid, stats)
        out[evaluate_rows] = prediction
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return out


if FIT_SHRINKAGE:
    _reference = MEMBERS[len(MEMBERS) // 2]
    print(f"[L5] OOF 적합 — {_reference['name']} 로 {len(FOLDS)}폴드", flush=True)
    _oof = oof_predictions(_reference["settings"], _reference["epochs"], _reference["seed"])
    _rows = np.isfinite(_oof).all(axis=1)
    _centered_prediction = _oof[_rows] - TRAIN_CLIMATOLOGY
    _centered_target = train_targets[_rows] - TRAIN_CLIMATOLOGY
    _covariance = np.array([float(np.mean(_centered_prediction[:, h] * _centered_target[:, h]))
                            for h in range(12)])
    _variance = np.array([float(np.mean(_centered_prediction[:, h] ** 2)) for h in range(12)])
    LAM_SINGLE = _covariance / np.maximum(_variance, 1e-9)          # 멤버 1개짜리 λ

    # ---- 멤버 1개 -> K개 평균으로 옮긴다 (이 보정이 없으면 반드시 과수축한다) ----
    # OOF 는 멤버 하나의 예측이지만, 실제로 보정할 대상은 K개 평균이라 분산이 작다.
    #   Var(p_1) = V + σ²,  Var(p_K) = V + σ²/K,  Cov(p, y) = C  (잡음은 y 와 무관)
    #   λ_1 = C / (V + σ²) = C / A        ->      λ_K = C / (A − σ²(1 − 1/K))
    # σ² 는 멤버간 예측 분산으로 잰다. 설정 차이까지 섞여 있어 σ² 는 과대추정되고,
    # 그 방향은 λ_K -> 1, 즉 **보정을 덜 하는 쪽**이라 안전하다.
    _k = len(TEST_PREDICTIONS)
    _sigma2 = (np.var(np.stack(TEST_PREDICTIONS), axis=0, ddof=1).mean(axis=0)
               if _k > 1 else np.zeros(12))
    _denominator = np.maximum(_variance - _sigma2 * (1.0 - 1.0 / _k), 0.25 * _variance)
    LAM_H = np.clip(LAM_SINGLE * _variance / np.maximum(_denominator, 1e-9), *SHRINK_CLIP)

    print(f"[L5] 멤버1 λ  = {np.round(np.clip(LAM_SINGLE, 0, 2), 3).tolist()}")
    print(f"[L5] 멤버{_k} λ = {np.round(LAM_H, 3).tolist()}   <- 실제로 쓰는 값")
    print(f"     멤버간 분산 σ = {np.sqrt(_sigma2).mean():.1f} km/s, "
          f"단일 예측 표준편차 = {np.sqrt(_variance).mean():.1f} km/s")
    if float(np.mean(LAM_H <= SHRINK_CLIP[0] + 1e-9)) > 0.5:
        print("     ⚠️ λ 가 하한에 붙었다 = 예측이 타깃 대비 과분산이라는 뜻이다. "
              "S4 가 S3 보다 나쁘면 이 항부터 뺄 것.")
    print("     1 에 가까우면 앙상블이 이미 분산을 잘 줄였다는 뜻이고, 그것도 정보다.")
else:
    LAM_H = np.ones(12, np.float64)
    print("[L5] 건너뜀 (FIT_SHRINKAGE=False)")


def finalize(prediction):
    """수축 보정 + 물리 범위 클립."""
    shrunk = TRAIN_CLIMATOLOGY[None, :] + LAM_H[None, :] * (prediction - TRAIN_CLIMATOLOGY)
    return np.clip(shrunk, 200.0, 1200.0)

[L5] 건너뜀 (FIT_SHRINKAGE=False)


### 제출물 저장

In [14]:
# ============================  제출물 저장  ================================
FINAL_TEST = finalize(TEST_MEAN)
FINAL_VAL = finalize(VAL_MEAN)
_final_val_score, _final_val_horizon = official_rmse(val_targets, FINAL_VAL)

P3_PER_HORIZON = np.array([28.35, 44.22, 53.62, 59.68, 64.32, 68.00,
                           70.64, 72.72, 74.44, 76.22, 78.20, 80.02])
print(f"validation (참고용, 선택에 쓰지 않는다)  P3 {P3_PER_HORIZON.mean():.3f} -> "
      f"P11-{STAGE} {_final_val_score:.3f}")
print(pd.DataFrame({"horizon": HORIZONS, "P3": P3_PER_HORIZON,
                    "P11": _final_val_horizon.round(2),
                    "delta": (_final_val_horizon - P3_PER_HORIZON).round(2),
                    "persistence": VAL_PERSISTENCE.round(2),
                    "climatology": VAL_CLIMATOLOGY.round(2)}).to_string(index=False))

submission = pd.DataFrame(FINAL_TEST, columns=TARGET_COLUMNS)
submission.insert(0, "sample_id", test_inputs.sample_id.tolist())
assert submission.shape == (len(test_inputs), 13) and np.isfinite(FINAL_TEST).all()
submission.to_csv(SUBMISSION_DIR / "submission.csv", index=False)

torch.save({
    "code_version": "p11", "stage": STAGE,
    "members": MEMBER_STATES,
    "member_stats": MEMBER_STATS,
    "configs": [{"name": m["name"], "settings": m["settings"],
                 "seed": m["seed"], "epochs": m["epochs"]} for m in MEMBERS],
    "lam_h": np.asarray(LAM_H),
    "train_climatology": TRAIN_CLIMATOLOGY,
    "ensemble": "equal weight mean, 적합 파라미터 0개",
    "initialization": "random_from_scratch",
    "val_rmse": _final_val_score,
}, SUBMISSION_DIR / "model.pth")

print(f"\nsubmission.csv {submission.shape} / model.pth 멤버 {len(MEMBER_STATES)}개 저장")
print(submission[TARGET_COLUMNS].describe().loc[["mean", "std", "min", "max"]].round(1))

validation (참고용, 선택에 쓰지 않는다)  P3 64.203 -> P11-S1 65.622
 horizon    P3   P11  delta  persistence  climatology
       6 28.35 29.33   0.98    30.129999    94.190002
      12 44.22 45.67   1.45    49.610001    94.540001
      18 53.62 55.40   1.78    63.040001    94.769997
      24 59.68 61.82   2.14    73.260002    95.089996
      30 64.32 66.80   2.48    82.029999    95.199997
      36 68.00 70.52   2.52    89.150002    95.449997
      42 70.64 73.04   2.40    94.699997    95.570000
      48 72.72 74.69   1.97    99.059998    95.639999
      54 74.44 75.96   1.52   102.730003    95.669998
      60 76.22 77.12   0.90   105.949997    95.680000
      66 78.20 78.12  -0.08   108.660004    95.709999
      72 80.02 78.99  -1.03   110.930000    95.790001

submission.csv (3868, 13) / model.pth 멤버 6개 저장
      target_00  target_01  target_02  target_03  target_04  target_05  \
mean      408.3      409.4      410.3      411.1      412.2      413.0   
std        82.0       76.7       73.0       7

### 재현성 검증

In [15]:
# ======================  재현성 검증 (제출 전 필수)  =======================
# 규정: "제출된 코드로 다시 추론했을 때의 결과와 실제 제출 결과의 성능이 크게
# 차이 나는 경우 불이익". model.pth 만으로 submission.csv 가 재구성되는지 확인한다.

blob = torch.load(SUBMISSION_DIR / "model.pth", map_location="cpu", weights_only=False)
reproduced = []
for config, state, stats in zip(blob["configs"], blob["members"], blob["member_stats"]):
    configure(**config["settings"])
    model = build_model(stats)
    model.load_state_dict(state)
    model.to(DEVICE)
    prediction, _ = predict_split(model, test_inputs, test_index_matrix, test_wind,
                                  test_wind_valid, test_grid, stats)
    reproduced.append(prediction)
    del model
    gc.collect()

reproduced_mean = np.clip(
    blob["train_climatology"][None, :]
    + np.asarray(blob["lam_h"])[None, :]
    * (np.mean(reproduced, axis=0) - blob["train_climatology"]), 200.0, 1200.0)
saved = pd.read_csv(SUBMISSION_DIR / "submission.csv")[TARGET_COLUMNS].to_numpy()
gap = float(np.sqrt(np.mean((reproduced_mean - saved) ** 2)))
print(f"재현 오차 {gap:.6f} RMS km/s  ({'통과' if gap < 0.01 else '🔴 실패 — 원인 확인'})")
assert gap < 0.01, "model.pth 로 submission.csv 가 재현되지 않는다"

shared=224 head_input=235 (gather 0, ballistic 3, flags 0)
shared=224 head_input=235 (gather 0, ballistic 3, flags 0)
shared=224 head_input=235 (gather 0, ballistic 3, flags 0)
shared=288 head_input=464 (gather 96, ballistic 72, flags 0)
shared=288 head_input=464 (gather 96, ballistic 72, flags 0)
shared=288 head_input=464 (gather 96, ballistic 72, flags 0)
재현 오차 0.000000 RMS km/s  (통과)


## 제출 점검

In [16]:
# ============================  제출 점검  ==================================
# P9 의 점검 셀은 단일 모델의 FINAL_STATS 를 참조한다. P11 은 멤버가 여러 개라
# 그 변수가 없으므로, 규정이 요구하는 것만 직접 확인한다.

EXPECTED_TEST_ROWS = 3868

# 노트북 자신을 submission/code.ipynb 로 복사한다. Jupyter 는 실행 중인 파일 경로를
# 알려주지 않으므로 이름으로 찾는다. **저장(Ctrl+S) 한 뒤** 이 셀을 실행할 것.
NOTEBOOK_NAME = "code_p11.ipynb"
if Path(NOTEBOOK_NAME).exists():
    shutil.copyfile(NOTEBOOK_NAME, SUBMISSION_DIR / "code.ipynb")
    print(f"{NOTEBOOK_NAME} -> submission/code.ipynb 복사")
else:
    print(f"⚠️ {NOTEBOOK_NAME} 없음 — 노트북을 submission/code.ipynb 로 직접 저장할 것")

ok = True
for name in ["code.ipynb", "model.pth", "submission.csv"]:
    path = SUBMISSION_DIR / name
    if path.exists():
        print(f"  {name:16s} {path.stat().st_size / 1024 ** 2:8.2f} MiB")
    else:
        print(f"  {name:16s} 없음")
        ok = False

check = pd.read_csv(SUBMISSION_DIR / "submission.csv")
rows_ok = len(check) == EXPECTED_TEST_ROWS
columns_ok = list(check.columns) == ["sample_id"] + TARGET_COLUMNS
ids_ok = check.sample_id.tolist() == test_inputs.sample_id.tolist()
missing = int(check.isna().sum().sum())
values = check[TARGET_COLUMNS].to_numpy()
range_ok = bool((values >= 200.0).all() and (values <= 1200.0).all())
ok = ok and rows_ok and columns_ok and ids_ok and missing == 0 and range_ok

print(f"\n행 수      {len(check)} / 규정 {EXPECTED_TEST_ROWS}   {'OK' if rows_ok else '🔴'}")
print(f"컬럼       {'OK' if columns_ok else '🔴'} / sample_id 순서 {'OK' if ids_ok else '🔴'}")
print(f"결측       {missing}")
print(f"값 범위    {values.min():.1f} ~ {values.max():.1f} km/s  "
      f"{'OK' if range_ok else '🔴 물리 범위(200~1200) 밖'}")
print(f"멤버 수    {len(MEMBER_STATES)} (STAGE {STAGE})")
print("\n최종:", "통과" if ok else "실패 — 위 항목 확인")

code_p11.ipynb -> submission/code.ipynb 복사
  code.ipynb           0.11 MiB
  model.pth            5.90 MiB
  submission.csv       0.90 MiB

행 수      3868 / 규정 3868   OK
컬럼       OK / sample_id 순서 OK
결측       0
값 범위    267.2 ~ 790.3 km/s  OK
멤버 수    6 (STAGE S1)

최종: 통과
